# Installation of TDMClient


In [1]:
%pip install tdmclient

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


# Setup and configuration

## Import

Imports functions from the Python files.

In [2]:
import cv2
import numpy as np
import time
import math

import Interface as ui          # Calibration, snapshot and display
import MotionController as mc   # Thymio movements (turn_to / go_forward)
import Filtering as kf
import LocalNavigation as ln
import Vision as vi             # A* path planning (plan_path)

## Global constants
Define the global constants of the project.

In [3]:
ARENA_WIDTH_CM = 100       # To measure: distance (cm) between the outer
                            # corners of the left and right QR codes

GOAL_TOLERANCE_CM = 10.0     # Goal considered reached below this distance
SIMPLIFY_EPSILON_PX = 2.0   # Path simplification tolerance (map pixels)
MAX_STEP_CM = 5.0          # Maximum distance driven before measuring again
MIN_WAYPOINT_CM = 1.0       # Waypoints closer than this distance are skipped
FLUSH_FRAMES = 5            # Number of old frames discarded after a move
MAX_BLIND_SEGMENTS = 3

# Kidnapping: robot lifted then put down somewhere else
GROUND_LIFT_THR = 100       # prox.ground.delta below this = no ground (TO MEASURE: paper / black / in the air)
LIFT_STEPS = 4              # consecutive readings needed to validate lifted / put down (about 0.2 s)
PUT_DOWN_DELAY = 1.0        # wait (s) after the robot is put down; must be longer than one vision cycle

## Coordinates conversion methods
Methods defintion for coordinate conversion from map coordinates (pixels, y down) to world coordinates (cm, y up).

In [4]:
def px_to_cm():
    """Calculate the conversion factor from pixels to centimeters."""

    if ARENA_WIDTH_CM is None:
        raise ValueError(
            "Set ARENA_WIDTH_CM: width of the arena between "
            "the outer corners of the QR codes (cm)."
        )

    return ARENA_WIDTH_CM / ui.MAP_WIDTH


def to_world(p):
    """Convert a map point (pixels) into world coordinates (cm)."""

    s = px_to_cm()

    return (
        p[0] * s,
        -p[1] * s
    )


def heading_world(theta):
    """
    Convert the camera heading into the robot's world heading.

    Camera convention: red -> green, with y pointing down.
    World convention: mathematical coordinates, with y pointing up.
    """

    return -(theta + ui.HEADING_OFFSET)

def camera_pose_m(state):
    x_cm, y_cm = to_world(state["thymio"])
    return np.array([x_cm / 100, y_cm / 100, heading_world(state["theta"])])

def distance(a, b):
    """Calculate the Euclidean distance between two points."""

    return float(
        np.hypot(
            b[0] - a[0],
            b[1] - a[1]
        )
    )


## Global navigation planning methods
Methods definition for global navigation planning.

In [5]:
def simplify(path):
    """
    Simplify an A* path (one point per pixel)
    by keeping only the main corners.
    """

    pts = np.array(path, dtype=np.float32).reshape(-1, 1, 2)

    simplified = cv2.approxPolyDP(
        pts,
        SIMPLIFY_EPSILON_PX,
        False
    )

    # float(): approxPolyDP gives numpy.float32, which tdmclient's client.sleep() refuses
    return [
        (float(p[0]), float(p[1]))
        for p in simplified.reshape(-1, 2)
    ]


def next_target(position, path):
    """
    Determine the next target point.

    Select the first waypoint sufficiently far from the robot.
    If the waypoint is farther than MAX_STEP_CM, create an
    intermediate target at the maximum allowed distance.
    """

    waypoints = [
        to_world(w)
        for w in simplify(path)
    ]

    waypoint = next(
        (
            w for w in waypoints[1:]
            if distance(position, w) > MIN_WAYPOINT_CM
        ),
        waypoints[-1]
    )

    d = distance(position, waypoint)

    if d <= MAX_STEP_CM:
        return waypoint

    r = MAX_STEP_CM / d

    return (
        position[0] + r * (waypoint[0] - position[0]),
        position[1] + r * (waypoint[1] - position[1])
    )

## Camera visual display method
Definition method to display the camera visual with the rectified map. It returns the key pressed by the user.

In [6]:
def show(frame, state, H, H_inv):
    """
    Display the camera image and the rectified map.

    Returns the key pressed by the user.
    """

    # Display the original camera image
    camera_view = ui.draw_camera(
        frame.copy(),
        state,
        H_inv,
        stale=False
    )

    cv2.imshow(ui.WINDOW_CAMERA, camera_view)

    # Apply perspective transformation to obtain the map
    rectified = cv2.warpPerspective(
        frame,
        H,
        (ui.MAP_WIDTH, ui.MAP_HEIGHT)
    )

    # Display the rectified map
    map_view = ui.draw_map(
        rectified,
        state,
        stale=False
    )

    cv2.imshow(ui.WINDOW_MAP, map_view)

    return cv2.waitKey(1) & 0xFF


# Common algorithm

Calibration, then loop: snapshot -> display -> move to the next point -> new snapshot.

## Calibration

Camera configuration to correctly visualize our setup. The code will then try to identify the 4 QR codes and from that it will transform the visualized image into a 2D map splitted with pixels.

In [7]:
px_to_cm()   # check ARENA_WIDTH_CM before starting

cap = cv2.VideoCapture(ui.CAMERA_INDEX, cv2.CAP_DSHOW)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, ui.CAMERA_WIDTH)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, ui.CAMERA_HEIGHT)

True

# Algorithm's loop
## Camera calibration
This part fetches the video from the camera and splits it into different frames. If it can't open the camera, the code displays a message "Unable to open the camera". If the 4 QR codes are visibles, it says "Calibration... (show the 4 QR codes, q to quit)". Finally, it the user presses on "q", then the calibration is aborted. If the calibration has been entirely done and is ready to be used, then the code displays "Calibration done.".

## Path planning
In this part, one call the path planning algorithm and take the first iteration. We then give the position of the next step to the filtering methods that will apply a Kalman filter in order to output the angular and translational commands to the motion controller method. This is the part for the global navigation. However, if the local navigation is needed, then it changes of loop and start acting with the local navigation methods defined.

## Iterative motion
Both these 2 steps are repeated iteratively until the final position is reached.

In [ ]:
import asyncio

def vision_job():
    for _ in range(FLUSH_FRAMES):
        cap.read()
    ret, frame = cap.read()
    if not ret:
        return None, None
    return frame, ui.take_snapshot(frame)

node = None
try:
    if not cap.isOpened():
        raise RuntimeError("Unable to open the camera")

    # 1. Calibration (once)
    print("Calibration... (show the 4 QR codes, q to quit)")
    calibration = ui.calibrate_camera(cap)
    if calibration is None:
        raise RuntimeError("Calibration aborted ('q' pressed)")
    H, H_inv = calibration
    print("Calibration done.")

    # 2. Initial state (this was missing)
    k_state, P = None, None
    last_path = None
    was_local = False      # navigation mode at the previous step
    lifted = False         # robot currently in the air (kidnapping)
    low_count = high_count = 0
    t_put_down = -math.inf # time of the last put down
    left = right = 0

    node = await mc.open_node()
    await node.wait_for_variables({"prox.horizontal", "prox.ground.delta"})   # make sure the sensors are being read
    nav_mem = ln.new_local_nav_memory()
    loop = asyncio.get_running_loop()
    job = loop.run_in_executor(None, vision_job)     # first vision job
    t_prev = time.perf_counter()

    while True:
        now = time.perf_counter()
        step = now - t_prev
        t_prev = now

        # kidnapping: the 2 ground sensors see no ground when the robot is lifted
        g = list(node.v.prox.ground.delta)
        if not lifted:
            low_count = low_count + 1 if max(g) < GROUND_LIFT_THR else 0
            if low_count >= LIFT_STEPS:             # lifted: stop and forget everything
                print("Robot lifted: stop")
                lifted, high_count = True, 0
                k_state, P = None, None
                last_path = None
                nav_mem = ln.new_local_nav_memory()
                was_local = False
                left = right = 0
        else:
            high_count = high_count + 1 if min(g) > GROUND_LIFT_THR else 0
            if high_count >= LIFT_STEPS:            # put down: restart after PUT_DOWN_DELAY
                print("Robot put down: restart")
                lifted, low_count = False, 0
                t_put_down = now

        # paused while lifted and during PUT_DOWN_DELAY: the frames taken meanwhile are ignored
        paused = lifted or (now - t_put_down < PUT_DOWN_DELAY)

        # prediction with the commands sent during the last step
        if k_state is not None and not paused:
            k_state, P = kf.kalman(k_state, P, left, right, step, None)

        # camera correction, only when a new result is ready
        if job.done():
            frame, state = job.result()
            job = loop.run_in_executor(None, vision_job)
            if state is not None:
                if state["thymio"] is not None and not paused:
                    z = camera_pose_m(state)
                    if k_state is None:
                        k_state, P = z, kf.R.copy()
                    else:
                        k_state, P = kf.kalman(k_state, P, 0, 0, 0, z)
                    # A* only when no path is being followed (start, or end of a local navigation)
                    if last_path is None:
                        last_path = vi.plan_path(state["grid"], *state["thymio"], state["goal"])
                state["path"] = last_path      # display the path being followed
                if show(frame, state, H, H_inv) == ord('q'):
                    break

        if paused:
            left = right = 0
            await mc.set_motors(node, 0, 0)
            await mc.client.sleep(0.05)
            continue

        if k_state is None or last_path is None:
            left = right = 0
            await mc.set_motors(node, 0, 0)
            await mc.client.sleep(0.05)
            continue

        # control, at every step
        x, y, theta = k_state[0] * 100, k_state[1] * 100, k_state[2]
        if distance((x, y), to_world(last_path[-1])) < GOAL_TOLERANCE_CM:
            print("Goal reached!")
            break
        # cut the beginning of the path up to the point closest to the robot (never go back)
        i = min(range(len(last_path)), key=lambda k: distance((x, y), to_world(last_path[k])))
        last_path = last_path[i:]
        target = next_target((x, y), last_path)

        # heading error to the target, wrapped to [-pi, pi]
        alpha = math.atan2(target[1] - y, target[0] - x) - theta
        alpha = math.atan2(math.sin(alpha), math.cos(alpha))

        # local navigation has priority when an obstacle is close
        p = list(node.v.prox.horizontal)             # 5 front sensors
        active, l, r = ln.local_nav_step(p, nav_mem, alpha)

        # end of the obstacle avoidance: new A* from the current position at the next frame
        if was_local and not active:
            last_path = None
        was_local = active

        if active:
            left, right = int(l), int(r)
        else:
            v, w = mc.astolfi_controller(x, y, theta, target)
            left, right = mc.v_w_to_motors(v, w)

        await mc.set_motors(node, left, right)
        await mc.client.sleep(mc.dt)
finally:
    if node is not None:
        await mc.set_motors(node, 0, 0)
        await node.unlock()
    cap.release()
    cv2.destroyAllWindows()
    print("Stopped: motors off, camera closed.")

Calibration... (show the 4 QR codes, q to quit)
Calibration done.


# Final destination
## Windows all closed

Since the Thymio meets the final destination, the algorithms has come to an end and the windows are closed.